In [ ]:
# IMPORT LIBRARY
import numpy as np
import keras
import tensorflow as tf
from google.colab import drive
import matplotlib as plt
%matplotlib inline

In [ ]:
drive.mount('/content/drive')
# image_dir = '/content/drive/MyDrive/TDSC/gtzan_images'
image_dir = '/content/drive/MyDrive/TDSC/dataset_MFCC_1162_385_15 seconds_vers'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define image size and batch size
image_size = (128, 387)
batch_size = 32

# Create training dataset
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=image_size,
    batch_size=batch_size,
)

class_names = train_dataset.class_names

# Create validation dataset
val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    image_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=image_size,
    batch_size=batch_size,
)

Found 15991 files belonging to 10 classes.
Using 12793 files for training.
Found 15991 files belonging to 10 classes.
Using 3198 files for validation.


In [ ]:
# Normalize the dataset
def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_dataset = train_dataset.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# Optimize dataset performance
train = train_dataset.cache().shuffle(buffer_size=1000).prefetch(buffer_size=tf.data.AUTOTUNE)
validation = val_dataset.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
# Define ModelCheckpoint callback
checkpoint_filepath = f'best_crnn.keras'
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=False,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True
)

### Custom CRNN

In [ ]:
def create_model():
    model = tf.keras.models.Sequential([
        # Convolutional layers
        tf.keras.layers.Conv2D(16, (3, 3), dilation_rate=(1, 1), activation='relu', input_shape=(128, 387, 3)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv2D(32, (3, 3), dilation_rate=(1, 1), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Conv2D(32, (3, 3), dilation_rate=(1, 1), activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Dropout(0.2),

        # tf.keras.layers.Conv2D(64, (3, 3), dilation_rate=(16, 16), activation='relu'),
        # tf.keras.layers.BatchNormalization(),
        # tf.keras.layers.MaxPooling2D(2, 2),
        # tf.keras.layers.Dropout(0.2),

        # Recurrent layers
        tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten()),
        tf.keras.layers.LSTM(128),

        # Dense layers
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

    # Compile the model with the Adam optimizer, categorical crossentropy loss, and accuracy metric
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Return the compiled model
    return model

In [ ]:
model = create_model()
model.summary()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 126, 385, 16)        │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 126, 385, 16)        │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 63, 192, 16)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 63, 192, 16)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 61, 190, 32)         │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 61, 190, 32)         │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 30, 95, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 30, 95, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 28, 93, 32)          │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 28, 93, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 46, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 14, 46, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ (None, 14, 1472)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │         819,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          16,512 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           1,290 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 852,170 (3.25 MB)

 Trainable params: 852,010 (3.25 MB)

 Non-trainable params: 160 (640.00 B)

### Pretrained CRNN - MobileNetV2

In [ ]:
pretrained = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False)

# Set all layers in the pretrained model to be non-trainable
for layer in pretrained.layers:
    layer.trainable = False

pretrained.summary()

In [ ]:
# Use full layer 'out_relu'
# Get the output of the 'block_13_expand_relu' layer
last_layer = pretrained.get_layer('block_13_expand_relu')
last_output = last_layer.output

In [ ]:
def mobilenet():
    x = tf.keras.layers.Conv2D(32, (3, 3), dilation_rate=(2, 2), activation='relu', padding='same')(last_output)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D(2, 2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)

    # Recurrent layers
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten())(x)
    x = tf.keras.layers.LSTM(128)(x)

    # Dense layers
    x = tf.keras.layers.Dense(128)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

    model = tf.keras.Model(pretrained.input, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])
    return model

model = mobilenet()

### Pretrained CRNN - Xception

In [ ]:
pretrained = tf.keras.applications.Xception(input_shape=(128, 387, 3), include_top=False)

# Set all layers in the pretrained model to be non-trainable
for layer in pretrained.layers:
    layer.trainable = False

pretrained.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "xception"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 128, 387, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1_conv1 (Conv2D)     │ (None, 63, 193, 32)    │            864 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1_conv1_bn           │ (None, 63, 193, 32)    │            128 │ block1_conv1[0][0]     │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1_conv1_act          │ (None, 63, 193, 32)    │              0 │ block1_conv1_bn[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1_conv2 (Conv2D)     │ (None, 61, 191, 64)    │         18,432 │ block1_conv1_act[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1_conv2_bn           │ (None, 61, 191, 64)    │            256 │ block1_conv2[0][0]     │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1_conv2_act          │ (None, 61, 191, 64)    │              0 │ block1_conv2_bn[0][0]  │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2_sepconv1           │ (None, 61, 191, 128)   │          8,768 │ block1_conv2_act[0][0] │
│ (SeparableConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2_sepconv1_bn        │ (None, 61, 191, 128)   │            512 │ block2_sepconv1[0][0]  │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2_sepconv2_act       │ (None, 61, 191, 128)   │              0 │ block2_sepconv1_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2_sepconv2           │ (None, 61, 191, 128)   │         17,536 │ block2_sepconv2_act[0… │
│ (SeparableConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2_sepconv2_bn        │ (None, 61, 191, 128)   │            512 │ block2_sepconv2[0][0]  │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 31, 96, 128)    │          8,192 │ block1_conv2_act[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block2_pool               │ (None, 31, 96, 128)    │              0 │ block2_sepconv2_bn[0]… │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization  

 Total params: 20,861,480 (79.58 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 20,861,480 (79.58 MB)

In [ ]:
# Use full layer 'block14_sepconv2_act'
# Get the output of the 'add_10' layer
last_layer = pretrained.get_layer('block14_sepconv2_act')
last_output = last_layer.output

In [ ]:
def xception():
    # x = tf.keras.layers.Conv2D(32, (3, 3), dilation_rate=(2, 2), activation='relu', padding='same')(last_output)
    # x = tf.keras.layers.BatchNormalization()(x)
    # x = tf.keras.layers.MaxPooling2D(2, 2)(x)
    # x = tf.keras.layers.Dropout(0.2)(x)

    # Recurrent layers
    x = tf.keras.layers.TimeDistributed(tf.keras.layers.Flatten())(last_output)
    x = tf.keras.layers.LSTM(128)(x)

    # Dense layers
    x = tf.keras.layers.Dense(128)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

    model = tf.keras.Model(pretrained.input, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])
    return model

model = xception()

## MODEL FIT

In [ ]:
# Set numbers of epochs
EPOCHS = 30

# Train the model
history = model.fit(train,
                    epochs=EPOCHS,
                    verbose=1,
                    validation_data=validation,
                    callbacks=[model_checkpoint_callback])

Epoch 1/30
400/400 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.2661 - loss: 2.0325

In [ ]:
epochs_range = range(EPOCHS)

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

plt.pyplot.figure(figsize=(8, 8))
plt.pyplot.plot(epochs_range, acc, label='Training Accuracy')
plt.pyplot.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.pyplot.legend(loc='lower right')
plt.pyplot.title('Training and Validation Accuracy')

In [ ]:
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.pyplot.figure(figsize=(8, 8))
plt.pyplot.plot(epochs_range, loss, label='Training Loss')
plt.pyplot.plot(epochs_range, val_loss, label='Validation Loss')
plt.pyplot.legend(loc='upper right')
plt.pyplot.title('Training and Validation Loss')
plt.pyplot.show()

## MODEL EVALUATION

In [ ]:
# Load the best model
best_model = f'best_crnn.keras'
model.load_weights(best_model)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
# Predict the labels
val_preds = model.predict(validation)
val_pred_labels = np.argmax(val_preds, axis=1)

# Extract true labels from the validation dataset
val_labels = np.concatenate([y.numpy() for x, y in validation], axis=0)

# Create confusion matrix
conf_matrix = confusion_matrix(val_labels, val_pred_labels)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=class_names)
disp.plot(cmap=plt.cm.Blues, xticks_rotation='vertical')  # Set rotation to 'vertical' for better readability if needed
plt.pyplot.title('Confusion Matrix')
plt.pyplot.show()

In [ ]:
print('Classification Report: \n', classification_report(val_labels, val_pred_labels, target_names=class_names))